In [ ]:
print('hello money')

In [ ]:
# ------------- configure dataframe printing layout 
import pandas as pd
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_columns', None)
#------------------------------------------------

# This ONLY affects output presentation, NOT the underlying data type
# TO FORMAT COLUMNS to display data not in scientific format
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')



In [ ]:
# FILE PATH and SQL QUERY

import numpy as np
import duckdb
from pathlib import Path
import plotly.graph_objects as go
import plotly.express as px
from binance.client import Client
from datetime import datetime
import time

# file path

folder = Path.cwd() / '_inputs'/'_1_sol_monthly_aggts_2026'

# access the files
sorted_pqt_files = sorted(folder.glob('*.parquet'))
# file list
pqt_file = [pqt.as_posix() for pqt in sorted_pqt_files]

SYMBOL = "SOLUSDT"

print(SYMBOL)
print(folder.exists())

In [ ]:
# coin stats like average volume from BINANCE

client = Client()
klines = client.futures_klines(symbol=SYMBOL, interval='1d', limit=21)
closed_klines = klines[:-1]  # Exclude today's incomplete candle
# Index 4: Close Price, Index 5: Volume
volumes = [float(k[5]) for k in closed_klines]
prices = [float(k[4]) for k in closed_klines]
# ---------------------------------
vol_20_ma = sum(volumes[-20:]) / 20
price_20_ma = sum(prices[-20:]) / 20
notional_ma = vol_20_ma * price_20_ma

vol = (f"\n volume ma for {SYMBOL} : $ {notional_ma:,.2f} \n")

print(vol)

In [ ]:
# AGGTRADES RAW TABLE

# LOAD  aggregated DATA TO dataframe with  DUCKDB

conn = duckdb.connect(database=":memory:")

# Select your desired timeframe here: '1 Day', '1 Hour', '5 Minutes', '4 Hours', etc.
timeframe = '1 Day'

qry = f"""--sql
SELECT
    time_bucket(INTERVAL '{timeframe}', epoch_ms(CAST(transact_time AS BIGINT))) AS open_time,
    arg_min(price, transact_time) AS open,
    max(price) AS high,
    min(price) AS low,
    arg_max(price, transact_time) AS close,
    time_bucket(INTERVAL '{timeframe}', epoch_ms(CAST(transact_time AS BIGINT))) + INTERVAL '{timeframe}' - INTERVAL '1 millisecond' AS close_time,
    sum(CASE WHEN is_buyer_maker = FALSE THEN (price * quantity) ELSE 0.0 END) AS buyVol_usdt,
    sum(CASE WHEN is_buyer_maker = TRUE  THEN (price * quantity) ELSE 0.0 END) AS sellVol_usdt
FROM read_parquet({pqt_file})
GROUP BY 1
ORDER BY open_time ASC;
"""

df = conn.execute(qry).df()
conn.close()

# print(df.tail())

In [ ]:
# RAW market MEGA METRICS TABLE  PART 1 (14 columns sofar)

# derived columns  ------------------------------------- PERCENTAGES
df['totalVol_usdt'] = (df['buyVol_usdt'] + df['sellVol_usdt']).round(3)
df['delta'] = (df['buyVol_usdt'] - df['sellVol_usdt']).round(3)

df['prc_change_window'] = ((
    df['close']-df['open'])/df['open'])*100

# df['full__pct_change']
# Candle direction condition (True for Bullish, False for Bearish)
is_bull = df['close'] >= df['open']
df['full_pct_change'] = np.where(
    is_bull,
    ((df['high'] - df['low']) / df['low']) * 100,
    ((df['low'] - df['high']) / df['high']) * 100
)

# Range as a percentage of the Open price
df['bar_volatility'] = ((df['high'] - df['low']) / df['open']) * 100

# core derived columns
df['s_delta'] = (df['delta']/notional_ma)*100

# df.to_parquet(file_output_path)


In [ ]:
# PART 2 ()  ADDING THE PIVOT POINTS of candle wicks TO THE TABLE ==================

left_len =10
right_len = 10
window_size = left_len + right_len + 1
#Rolling max/min centered on the target candle
rolling_max = df['high'].rolling(window= window_size, center=True).max()
rolling_min = df['low'].rolling(window= window_size, center= True).min()
# Indentify pivot points (hing must equal max of the window and low must equal min of the window )
df['pivot_high'] = df['high'].where(df['high']==rolling_max, None)
df['pivot_low'] = df['low'].where(df['low']==rolling_min, None)
# Combined column showing the price level at the swing point, or None otherwise
df['swing_point_wcks'] = df['pivot_high'].combine_first(df['pivot_low'])

#-------------------------------------------------------------------------------
#  Structural Pivot Detection with Body-Level Price Extraction
df['pivot_h_raw'] = df['high'].where(df['high'] == rolling_max, None)
df['pivot_l_raw'] = df['low'].where(df['low'] == rolling_min, None)
#  Extract true Body Top and Body Bottom levels across all candles
body_top = df[['open', 'close']].max(axis=1)
body_bottom = df[['open', 'close']].min(axis=1)
#  Read off Body Top/Bottom ONLY on candles identified as structural pivots
df['pivot_high_bdy'] = np.where(df['pivot_h_raw'].notna(), body_top, None)
df['pivot_low_bdy'] = np.where(df['pivot_l_raw'].notna(), body_bottom, None)
#  Combine into final swing point column (body-based price level)
df['swing_point_bdy'] = df['pivot_high_bdy'].combine_first(df['pivot_low_bdy'])
# Clean up temporary raw detector columns if desired
df = df.drop(columns=['pivot_h_raw', 'pivot_l_raw'])
#-------------------------------------------------------------------------------

In [ ]:
# PART 3  () computing internal metrics of SWING MOVES 

# 1. Create a dynamic segment group ID for each swing move
df['swing_group_wck'] = df['swing_point_wcks'].notna().cumsum()
df['swing_group_wck'] = df['swing_group_wck'].shift(1, fill_value=0)
# 2. Cumulative volume and delta metrics within each swing segment
df['cum_buyVol_internal'] = df.groupby('swing_group_wck')['buyVol_usdt'].cumsum()
df['cum_sellVol_internal'] = df.groupby('swing_group_wck')['sellVol_usdt'].cumsum()
df['cum_totalVol_internal'] = df.groupby('swing_group_wck')['totalVol_usdt'].cumsum()
df['cum_delta_internal'] = df.groupby('swing_group_wck')['delta'].cumsum()


#  detailed column ::: PERCENTAGE CHANGE FROM SWING PIVOT POINT

# 1. Generate Swing Groups strictly based on body pivot occurrences
df['is_pivot'] = df['swing_point_bdy'].notna() #has bo0lean results true/false
# Group 0 = pre-pivot candles. Group 1 = first swing move, Group 2 = second swing move, etc.
df['swing_group_bdy'] = df['is_pivot'].cumsum()
# 2. Extract the TRUE starting anchor price for each group
# ***
# For any group > 0, the first row of that group contains the pivot candle itself,
# so transform('first') picks up that exact pivot price!
df['pivot_anchor'] = df.groupby('swing_group_bdy')['swing_point_bdy'].transform('first')
# -------
# 3. Handle Group 0 (Synthetic Baseline) without variable collisions
first_candle_open = float(df['open'].iloc[0])
first_candle_close = float(df['close'].iloc[0])
# Safe extraction of body top / body bottom using Series min/max (avoids builtin max/min collision)
body_top_first = max([first_candle_open, first_candle_close])
body_bottom_first = min([first_candle_open, first_candle_close])
# Get the first valid pivot price in the whole dataset
valid_anchors = df['pivot_anchor'].dropna()

if len(valid_anchors) > 0:
    first_valid_pivot = valid_anchors.iloc[0]
    # If first pivot is above 1st candle open -> Bullish move expected -> anchor to body_bottom
    # If first pivot is below 1st candle open -> Bearish move expected -> anchor to body_top
    pre_swing_start_price = body_bottom_first if first_valid_pivot > first_candle_open else body_top_first
else:
    pre_swing_start_price = first_candle_open
# Fill Group 0's NaN with the pre-pivot synthetic price
df['swing_start_price'] = df['pivot_anchor'].fillna(pre_swing_start_price)
# ---------------------------------------------------------
# 4. Calculate Price Percentage change from swing Pivot Point
# ---------------------------------------------------------
df['pct_change_from_pivot'] = ((df['close'] - df['swing_start_price']) / df['swing_start_price']) * 100

# Clean up temporary detection columns
df = df.drop(columns=['is_pivot', 'pivot_anchor'])

#===========================================================

df['cum_delta_all'] = df['delta'].cumsum()
# df = df.drop(columns=['cum_delta_all'])

# MORE COLUMNS TO DROP
df = df.drop(columns=['swing_group_wck', 'swing_group_bdy','swing_start_price'])

In [ ]:
# PART 4 () - OTHER 2 analysis columns 

#-----------------------#  POTENTIAL PIVOT COLUMN ---------

# Calculate rolling 10-bar high and low of the PREVIOUS 10 candles (excluding current candle)
prev_left_max = df['high'].shift(1).rolling(window=left_len).max()
prev_left_min = df['low'].shift(1).rolling(window=left_len).min()
# Conditions for potential pivots
is_potential_short = df['high'] > prev_left_max  # New 10-bar high (Potential 'SHORT')
is_potential_long = df['low'] < prev_left_min   # New 10-bar low (Potential 'LONG')
# Create df['potential_pivot'] column
# Labels: 'S' for potential short pivot, 'L' for potential long pivot, None otherwise
df['potential_pivot'] = np.select([is_potential_short, is_potential_long], ['S', 'L'], default=None)

#-----------------------#  ACTUAL_PIVOT COLUMN---------------------

# 1. Label confirmed pivot highs ('S') and confirmed pivot lows ('L')
df['actual_pivot_h'] = np.where(df['pivot_high'].notna(), 'S', None)
df['actual_pivot_l'] = np.where(df['pivot_low'].notna(), 'L', None)

# 2. Combine into a single df['actual_pivot'] column
df['actual_pivot'] = df['actual_pivot_h'].combine_first(df['actual_pivot_l'])

# Optional: Drop the intermediate temporary columns
df = df.drop(columns=['actual_pivot_h', 'actual_pivot_l'])

# ==================================================================================

# Get individual ROWS and COLUMNS counts
rows = df.shape[0]
columns = df.shape[1]
print(f"Rows: {rows}, Columns: {columns} \n")

# print(df.head(50))
# print(df[:2].to_string())
# print('\n')

# print(df[['open_time', 'prc_change_window','pivot_high','pivot_low','swing_point_wcks','swing_point_bdy','delta','cum_delta_internal','cum_delta_all','potential_pivot','actual_pivot','pct_change_from_pivot']][:5].to_string())

# print(swing_start_price.to_string())

print(df[['open_time', 'potential_pivot', 'actual_pivot']][:10].to_string())

In [ ]:
# PLOTTING THE CANDLE GRAPH

fig1 = go.Figure()

fig1 = go.Figure(data=[
    go.Candlestick(
        x=df['open_time'],
        open=df['open'],
        high=df['high'],
        low=df['low'],
        close=df['close'],
        name='Price',
        # bullish candles
        increasing_line_color='white',
        increasing_fillcolor='rgba(0, 0, 0, 0)',
        increasing_line_width=1,
        # bearish candles
        decreasing_line_color='#0677d4',
        decreasing_fillcolor='rgba(0, 0, 0, 0)',
        decreasing_line_width=1
    )
])
# -------------------------------------------- delta
# fig1.add_trace(go.Scatter(x=df.open_time, y=df.delta,
#               mode='lines', line=dict(color='purple', width=1)))

# -------------------------------------------- cum_delta_internal
# fig1.add_trace(go.Scatter(x=df.open_time, y=df.cum_delta_internal,
#               mode='lines', line=dict(color='blue', width=1)))

# -------------------------------------------- cum_delta_all
# fig1.add_trace(go.Scatter(x=df.open_time, y=df.delta,
#               mode='lines', line=dict(color='pink', width=1)))

# -------------------------------------------- pct_change_from_pivot
# fig1.add_trace(go.Scatter(x=df.open_time, y=df.pct_change_from_pivot,
#               mode='lines', line=dict(color='yellow', width=1)))

# ---------------------------------------swing points line
fig1.add_trace(go.Scatter(x=df.open_time, y=df.swing_point_bdy,
                         connectgaps= True,
              mode='lines', line=dict(color='green', width=1)))

# ----------------------------------------------
fig1.update_layout(title='sol 2026 jan to june  day timeframe',
                  template='plotly_dark',
                  xaxis_rangeslider_visible=False,
                  hovermode='x unified',
                  showlegend=False,
                  height=600
                  )

# Add Pivot High Markers
pivot_highs = df.dropna(subset=['pivot_high_bdy'])
fig1.add_trace(go.Scatter(
    x=pivot_highs['open_time'],
    y=pivot_highs['pivot_high_bdy'],
    mode='markers+text',
    marker=dict(symbol='triangle-down', size=10, color='red'),
    text=pivot_highs['pivot_high_bdy'],
    textposition='top center',
    name='Pivot High bdy'
))

# Add Pivot Low Markers
pivot_lows = df.dropna(subset=['pivot_low_bdy'])
fig1.add_trace(go.Scatter(
    x=pivot_lows['open_time'],
    y=pivot_lows['pivot_low_bdy'],
    mode='markers+text',
    marker=dict(symbol='triangle-up', size=10, color='green'),
    text=pivot_lows['pivot_low_bdy'],
    textposition='bottom center',
    name='Pivot Low bdy'
))

In [ ]:
# SWING PIVOT POINTS TABLE ---  PART A

# DF for PIVOT POINTS ONLY ================================================================

pv_df = df[df['swing_point_wcks'].notna()]
# print(pv_df.head())

# 1. filter out only the candles that formed valid pivots
pivots = df[df['swing_point_wcks'].notna()].copy()
# 2. determine pivot type (high or low ) for each pivot point
pivots['type']= np.where(pivots['pivot_high'].notna(), 'high', 'low')
# 3. Reset index to calculate exact bar index distance between pivots
pivots = pivots.reset_index().rename(columns={'index':'orig_bar_index'})

# BUILD THE SWING POINTS TABLE (spt)  ========================================================
spt = pd.DataFrame()

# Start & End attributes per move (shift by -1 to pair consecutive pivots)
spt['swing_id'] = range(1, len(pivots))
spt['swing_start_time'] = pivots['open_time'][:-1].values
spt['swing_start_price']   = pivots['swing_point_wcks'][:-1].values
# spt['swing_end_time'] = pivots['open_time'][1:].values
spt['swing_end_time'] = pivots['close_time'][1:].values  
# spt =spt.drop(columns=['swing_end_time'])
spt['swing_end_price']   = pivots['swing_point_wcks'][1:].values
#bars elapsed between start and end of the swing move
spt['bars'] = (pivots['orig_bar_index'][1:].values - pivots['orig_bar_index'][:-1].values)
#percentage price change
spt['price_change_pct'] =  ((spt['swing_end_price']- spt['swing_start_price'])/spt['swing_start_price'])*100
spt['abs_price_pct']=spt['price_change_pct'].abs()

#---------------------- SWING BODY PERCENTAGE CHANGE -----------------
spt['swing_start_body_price']   = pivots['swing_point_bdy'][:-1].values
spt['swing_end_body_price']   = pivots['swing_point_bdy'][1:].values
spt['price_pct_bdy'] =  ((spt['swing_end_body_price']- spt['swing_start_body_price'])/spt['swing_start_body_price'])*100
spt['abs_price_pct_bdy'] = spt['price_pct_bdy'].abs()

# spt = spt.drop(columns=['swing_start_body_price','swing_end_body_price'])

#----------------------------------
# spt['pct_change_internal_orig'] = pivots['cum_pct_change_internal']
# spt["pct_change_internal"] = spt["pct_change_internal_orig"].shift(-1)
# spt.drop(columns=['pct_change_internal_orig'])
# spt['pct_chg_internal_abs'] = pivots['cum_pct_change_internal'].abs()

#duration of the swing move
spt['duration']= (spt['swing_end_time']- spt['swing_start_time'])
spt =spt.drop(columns=['swing_end_time'])

#swing direction ; up or down
spt['direction']= np.where(spt['swing_end_price']>spt['swing_start_price'], 'up', 'down')

#Trend classification (bullish, bearish , ranging)
conditions =[spt['price_change_pct']>1.0, # bullish when change greater than 1%
             spt['price_change_pct']<-1.0 # bearish   
            ]
choices =['bullish', 'bear']

spt['trend']= np.select(conditions, choices, default='range')


# print(pivots.head())
# print(spt[150:].to_string())

# print(spt.head())

In [ ]:
 #  spt  PART B
# Extract bar start and end indices for every swing move from the pivots dataframe
start_indices = pivots['orig_bar_index'][:-1].values
end_indices = pivots['orig_bar_index'][1:].values

# Extract volume arrays from original dataframe (df)
df_buy_vol = df['buyVol_usdt'].values
df_sell_vol = df['sellVol_usdt'].values

# Use np.add.reduceat to compute cumulative sum between pivot indices (inclusive of start, up to end)

spt['swing_buyVol_usdt'] = [
    df_buy_vol[s + 1 : e + 1].sum() for s, e in zip(start_indices, end_indices)
]
spt['swing_sellVol_usdt'] = [
    df_sell_vol[s + 1 : e + 1].sum() for s, e in zip(start_indices, end_indices)
]

# Total volume
spt['swing_total_vol'] = spt['swing_buyVol_usdt'] + spt['swing_sellVol_usdt']

#  Volume Delta 
spt['swing_delta_usdt'] = spt['swing_buyVol_usdt'] - spt['swing_sellVol_usdt']

# spt['cum_swing_delta'] =spt['swing_delta_usdt'].cumsum()
# spt['swing_s_delta'] = (spt['swing_delta_usdt']/notional_ma)*100
# temporarily remove the spt['swing_s_delta'] column from 'spt' for now
# spt = spt.drop(columns=['swing_s_delta'])

# SWING STRENGTH COLUMNS
#-----------------------------------#1 VOLUME EFFICIENCY --------------
# Net taker (Delta ) imbalance ratio (0.0 to 1.0)
# spt['delta_imbalance']=(spt['swing_delta_usdt'] ).abs() / spt['swing_total_vol']

spt['delta_imbalance_pct'] = ((spt['swing_delta_usdt'] ).abs() / spt['swing_total_vol'])*100
# Volume Efficiency score: Price percentage move per unit of delta imbalance
spt['efficiency_score'] = spt['price_change_pct'].abs() / (spt['delta_imbalance_pct'] + 1e-6)
# Threshold: Above median efficiency = Strong, below = Weak
median_eff = spt['efficiency_score'].median()
spt['strength_1'] = np.where(spt['efficiency_score'] >= median_eff, 'strong', 'weak')

#-----------------------------------#2 PRICE change VELOCITY ----------- for the swing move
# Absolute price percentage change per candle
spt['velocity'] = spt['price_change_pct'].abs() / spt['bars']
velocity_condition = spt['velocity'].quantile(0.25)
# spt['strength_2'] = np.where(spt['velocity'] >= spt['velocity'].quantile(0.25), 'strong', 'weak')
# -------------------------------- #3 ORDERFLOW DOMINANCE -----------------
spt['buy_ratio'] =(spt['swing_buyVol_usdt'] / spt['swing_total_vol'])*100

# Get individual ROWS and COLUMNS counts
rows = spt.shape[0]
columns = spt.shape[1]

# print(f"median_eff = {median_eff.round(2)} \n")

print(f"Rows: {rows}, Columns: {columns} \n")
print(spt.head(2), "\n")

# PRINT only row of the table i want 
# print(f"velocity 25th percentile = {velocity_condition.round(2)} % change per bar")
print(spt[['swing_id', 'swing_start_time','swing_start_price','swing_end_price','bars','price_change_pct','price_pct_bdy','trend','strength_1','swing_total_vol','swing_delta_usdt','delta_imbalance_pct','efficiency_score','velocity','buy_ratio']].head(5))



In [ ]:
#  A) -- DIRECTION statistics
# LAYER   1   ***  COLUMN ANALYSIS  -- spt['price_change_pct']

x =spt['price_pct_bdy']

#  original percentage change data
mean =          x.mean()
median =  x.median()
max_pct =        x.max()   # Highest price change percentage
min_pct =      x.min()  # Lowest price change percentage
standard_deviation = x.std()
variance = x.var()
percentile_25th =x.quantile(0.25)
percentile_50th =x.quantile(0.5)
percentile_75th =x.quantile(0.75)
percentile_90th =x.quantile(0.90)
percentile_95th =x.quantile(0.95)

# DISPLAY
print(f" === original percentage change data -- SYMBOL :{SYMBOL} -- Timeframe :{timeframe} ======")
print(f"Average Swing: {mean:.2f} %")
print(f"Median Swing:  {median:.2f} %")
print(f"Standard dev:  {standard_deviation:.2f} %")
# print(f"Variance    :  {variance:.2f} sq %²")
print(f"\nLowest Swing:  {min_pct:.2f} %")
print(f"Highest Swing:  {max_pct:.2f} %")
print(f"\n25th percentile:  {percentile_25th:.2f} %")
print(f"50th percentile:  {percentile_50th:.2f} %")
print(f"75th percentile:  {percentile_75th:.2f} %")
print(f"90th percentile:  {percentile_90th:.2f} %")
print(f"95th percentile:  {percentile_95th:.2f} %")

In [ ]:
#  B) -- MAGNITUDE  statistics
# LAYER   1   ***  accurate metrics

y =spt['abs_price_pct_bdy']

# ABSOLUTE   percentage change data

mean_2 =          y.mean()
median_2 =  y.median()
max_pct_2 =        y.max()   # Highest price change percentage
min_pct_2 =      y.min()   # Lowest price change percentage
standard_deviation_2 = y.std()
variance_2 = y.var()
percentile_25th_2 =y.quantile(0.25)
percentile_50th_2 =y.quantile(0.5)
percentile_75th_2 =y.quantile(0.75)
percentile_90th_2 =y.quantile(0.90)
percentile_95th_2 =y.quantile(0.95)
percentile_99th_2 =y.quantile(0.99)

# DISPLAY
print(f" === ABSOLUTE percentage change data -- SYMBOL :{SYMBOL} -- Timeframe :{timeframe} ======")
print(f"Average Swing: {mean_2:.2f} %")
print(f"Median Swing:  {median_2:.2f} %")
print(f"Standard dev:  {standard_deviation_2:.2f} %")
# print(f"Variance    :  {variance_2:.2f} sq %²")
print(f"\nLowest Swing:  {min_pct_2:.2f} %")
print(f"Highest Swing:  {max_pct_2:.2f} %")
print(f"\n25th percentile:  {percentile_25th_2:.2f} %")
print(f"50th percentile:  {percentile_50th_2:.2f} %")
print(f"75th percentile:  {percentile_75th_2:.2f} %")
print(f"90th percentile:  {percentile_90th_2:.2f} %")
print(f"95th percentile:  {percentile_95th_2:.2f} %")
print(f"99th percentile:  {percentile_99th_2:.2f} %")

In [ ]:
# PLOTTING THE CANDLE GRAPH

fig2 = go.Figure()


fig2 = go.Figure(data=[
    go.Candlestick(
        x=df['open_time'],
        open=df['open'],
        high=df['high'],
        low=df['low'],
        close=df['close'],
        name='Price',
        # bullish candles
        increasing_line_color='white',
        increasing_fillcolor='rgba(0, 0, 0, 0)',
        increasing_line_width=1,
        # bearish candles
        decreasing_line_color='#0677d4',
        decreasing_fillcolor='rgba(0, 0, 0, 0)',
        decreasing_line_width=1
    )
])
# -------------------------------------------- sma line
# fig.add_trace(go.Scatter(x=df.open_time, y=df.sma25,
#               mode='lines', line=dict(color='purple', width=1)))
# -------------------------------------------- delta_imbalance_pct


# ---------------------------------------swing points line
fig2.add_trace(go.Scatter(x=df.open_time, y=df.swing_point_bdy,
                         connectgaps= True,
              mode='lines', line=dict(color='green', width=1)))
# ----------------------------------------------
fig2.update_layout(title='sol 2026 jan to june  day timeframe',
                  template='plotly_dark',
                  xaxis_rangeslider_visible=False,
                  hovermode='x unified',
                  showlegend=False,
                  height=600
                  )

# Add Pivot High Markers
pivot_highs = df.dropna(subset=['pivot_high_bdy'])
fig2.add_trace(go.Scatter(
    x=pivot_highs['open_time'],
    y=pivot_highs['pivot_high_bdy'],
    mode='markers+text',
    marker=dict(symbol='triangle-down', size=10, color='red'),
    text=pivot_highs['pivot_high_bdy'],
    textposition='top center',
    name='Pivot High bdy'
))

# Add Pivot Low Markers
pivot_lows = df.dropna(subset=['pivot_low_bdy'])
fig2.add_trace(go.Scatter(
    x=pivot_lows['open_time'],
    y=pivot_lows['pivot_low_bdy'],
    mode='markers+text',
    marker=dict(symbol='triangle-up', size=10, color='green'),
    text=pivot_lows['pivot_low_bdy'],
    textposition='bottom center',
    name='Pivot Low bdy'
))

In [ ]:
# fig = go.Figure()

# fig.add_trace(go.Scatter(x=spt.swing_start_time, y=spt.delta_imbalance_pct,
#               mode='lines', line=dict(color='orange', width=1)))